# Toronto Airbnb Pricing Analytics
**Lambert Tan**

This notebook examines how property characteristics, location, and booking features are associated with nightly Airbnb prices in Toronto. It is the reproducible analysis behind the portfolio project and Streamlit scenario tool.

## 1. Research question

**After accounting for observable listing characteristics, which factors are most strongly associated with nightly price in Toronto?**

The emphasis is explanation rather than causal inference. The data are observational and cross-sectional, so the coefficients are interpreted as conditional associations.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_cleaning import clean_listings
from src.feature_engineering import engineer_features
from src.modeling import fit_final_model, coefficient_table, estimate_nightly_price


## 2. Load and audit the source data

The November 2025 Toronto detailed-listings snapshot contains 21,468 observations and 79 fields. Before modelling, I check the dimensions, missingness, and price field rather than treating the downloaded file as analysis-ready.

In [ ]:
raw = pd.read_csv(ROOT / 'data/raw/toronto_listings_detail_nov.csv')
print(f'Raw shape: {raw.shape}')
raw[['price', 'room_type', 'bedrooms', 'bathrooms_text']].head()


In [ ]:
missing = (raw.isna().mean().sort_values(ascending=False).head(15) * 100).round(1)
missing.to_frame('missing_pct')


## 3. Cleaning decisions

The cleaning stage is intentionally conservative. Prices are converted from formatted strings to numeric values and restricted to **$15–$900**. The purpose of this range is to reduce the influence of implausible entries and extreme luxury listings that represent a different pricing segment.

I retain **Entire home/apt** and **Private room** listings. Shared rooms and hotel rooms are excluded because their product structure differs enough that a single coefficient comparison would be difficult to interpret. Listings without usable bedroom, bathroom, host-start, or availability information are also removed according to the rules in `src/data_cleaning.py`.

These thresholds are modelling choices rather than universal definitions of a valid Airbnb listing. They should therefore be revisited if the analysis is applied to another market or period.

In [ ]:
clean = clean_listings(raw)
print(f'Clean shape: {clean.shape}')
clean[['price', 'room_type', 'accommodates', 'bedrooms', 'bathrooms']].describe(include='all').T


## 4. Price distribution and transformation

Nightly price is strongly right-skewed. Modelling the raw dollar outcome would give high-priced listings disproportionate influence and would make a constant-dollar interpretation less useful across the price range. I therefore use **log(price)** as the dependent variable.

In a log-linear model, a coefficient can be converted to an approximate percentage difference. For larger coefficients, I use the exact transformation `exp(beta) - 1`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(clean['price'], bins=50)
axes[0].set(title='Nightly price', xlabel='CAD', ylabel='Listings')
axes[1].hist(np.log(clean['price']), bins=50)
axes[1].set(title='Log nightly price', xlabel='log(CAD)', ylabel='Listings')
plt.tight_layout()
plt.show()


## 5. Feature engineering

The engineered variables are designed to remain interpretable. Amenity count summarizes the length of each listing's amenity list. Host experience is measured from `host_since` to the snapshot date. Binary variables identify entire homes, shared bathrooms, Superhosts, and instant booking.

For location, I calculate Haversine distance to **Union Station**. I use it as a fixed downtown reference point because it is central and reproducible. The measure represents straight-line distance, not transit or driving time, so it should be interpreted as a proxy for centrality rather than accessibility.

In [ ]:
data = engineer_features(clean)
print(f'Engineered shape: {data.shape}')
data.describe().T.round(2)


## 6. Model choice

I use OLS on log nightly price because the objective is to estimate relationships that can be explained to a business audience. A more flexible machine-learning model could improve predictive performance, but that is not the primary question here.

The broader candidate set included host experience and Superhost status. The final testing-down specification retains eight variables: accommodates, bedrooms, bathrooms, shared bathroom, entire home, distance to downtown, amenity count, and instant booking. Removing variables is based on the model-development process rather than an assumption that the omitted concepts never matter.

## 7. Holdout validation and heteroskedasticity

The data are split **70/30** into training and test samples. The holdout sample is used as a basic check that explanatory fit does not deteriorate sharply outside the estimation sample.

Because cross-sectional price data often have non-constant residual variance, I also run a Breusch–Pagan test. The test rejects homoskedasticity in this sample, so I report **HC3 robust standard errors** for inference. HC3 changes the estimated uncertainty around the coefficients; it does not change the OLS point estimates.

In [ ]:
model, robust_model, metrics, bp = fit_final_model(data)
results = coefficient_table(model, robust_model)

pd.Series(metrics, name='value').round(4)


In [ ]:
pd.Series(bp, name='value')


### Validation reading

Train and test R² are both about 0.62. The similarity matters more here than squeezing out a higher score: there is no large train/test gap suggesting that the reported fit is confined to the estimation sample. The remaining unexplained variation is also a reminder that nightly price depends on factors not represented in this snapshot.

## 8. Final estimates

In [ ]:
display_vars = [
    'is_entire_home', 'is_shared_bath', 'distance_to_downtown_km',
    'instant_bookable', 'amenity_count', 'accommodates', 'bedrooms', 'bathrooms'
]
results.loc[display_vars, ['coefficient', 'robust_pvalue', 'percent_impact']].round(3)


### Interpretation

Holding the included variables constant, an entire home is associated with approximately **43.8% higher nightly price** than a private room, while a shared bathroom is associated with roughly **21.0% lower price**. Each additional kilometre from the downtown reference point is associated with about **2.7% lower price**.

Instant booking and amenity count have smaller estimated relationships. This distinction is useful for pricing: operational settings may provide incremental differences, but they do not appear to substitute for the underlying property format.

These are conditional associations. They should not be interpreted as the causal price change from modifying a property.

## 9. Pricing scenario

The fitted equation can be used to explore a hypothetical listing. The function below exponentiates the predicted log price. That makes the output intuitive in dollars, but it also creates a **retransformation issue**: `exp(E[log(price)|X])` is not generally equal to `E[price|X]`.

For that reason, I treat the result as a model-implied scenario rather than an unbiased forecast of mean nightly price. A forecasting-focused extension would estimate a retransformation correction, such as Duan's smearing factor, from the training residuals.

In [ ]:
scenario = dict(
    accommodates=4, bedrooms=2, bathrooms=1, is_shared_bath=0,
    is_entire_home=1, distance_to_downtown_km=5, amenity_count=35,
    instant_bookable=1
)
scenario_price = estimate_nightly_price(model, **scenario)
print(f'Model-implied scenario: {scenario_price:,.0f} CAD per night')


## 10. Practical implications

A host using these results should first compare the listing with properties that have a similar room type, bathroom arrangement, capacity, and size. Downtown distance can then refine that benchmark. Amenities and instant booking are better treated as smaller adjustments than as primary pricing anchors.

Superhost status should be interpreted carefully. Its removal from the final price specification does not imply that the badge has no business value; it may affect trust, conversion, or occupancy, none of which is directly measured here.

## 11. Limitations and next step

This analysis uses one cross-sectional snapshot and therefore cannot establish causality or capture seasonal price changes. It also lacks direct measures of occupancy, booking conversion, event demand, travel time, and live competitor inventory. Accommodates, bedrooms, and bathrooms are correlated measures of property size, so their individual coefficients should not be read in isolation.

The extension I would prioritize is a panel of repeated listing snapshots combined with a demand or occupancy measure. That would allow the analysis to separate persistent property differences from time-varying market conditions and would make a dynamic pricing model more defensible.